# Notebook 30 — Active-learning sample selection

`ActiveSampleSelector` picks the most informative samples to label
from a pool of unlabelled data under a fixed budget. Three strategies:

- `uncertainty` — F1 BayesianPathwayGMM posterior entropy
- `diversity` — greedy k-center on pairwise distances
- `hybrid` — weighted combination of both

Research use only. Not for clinical decision-making.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture

from pathway_subtyping.active import ActiveSampleSelector

rng = np.random.default_rng(0)
n_per_class = 120
n_pathways = 10
cluster_means = rng.standard_normal((3, n_pathways)) * 1.3
cluster_ids = np.repeat(np.arange(3), n_per_class)
features = pd.DataFrame(
    cluster_means[cluster_ids] + rng.normal(0, 0.7, (len(cluster_ids), n_pathways)),
    columns=[f'PATH_{i}' for i in range(n_pathways)],
)
labels = pd.Series(cluster_ids, name='cluster')

## 1. Uncertainty-based selection

Fit a Bayesian GMM (or use F1's `BayesianPathwayGMM`) and feed the
posterior probabilities to the selector.

In [ ]:
gmm = GaussianMixture(n_components=3, random_state=0).fit(features.to_numpy())
probs = gmm.predict_proba(features.to_numpy())

result = ActiveSampleSelector(strategy='uncertainty').select(
    features=features, budget=30, probs=probs,
)
print(result.to_dict())

## 2. Diversity-based selection

Greedy k-center covers the feature space. Useful when you have no
trained model yet.

In [ ]:
result = ActiveSampleSelector(strategy='diversity', seed=0).select(
    features=features, budget=30,
)
print(result.to_dict())

## 3. Hybrid selection

Combine uncertainty + diversity. `alpha=0.5` weights them equally;
`alpha=1.0` is pure uncertainty, `alpha=0.0` is pure diversity.

In [ ]:
result = ActiveSampleSelector(strategy='hybrid', alpha=0.5).select(
    features=features, budget=30, probs=probs,
)
print(result.to_dict())

## 4. Roadmap acceptance: 90% accuracy at 40% labels

Split into pool + test, run active selection at 40% budget, measure
classification accuracy. Should reach >= 90% of the full-cohort
accuracy.

In [ ]:
def nn_accuracy(train_X, train_y, test_X, test_y):
    tn = np.linalg.norm(train_X, axis=1, keepdims=True); tn[tn == 0] = 1.0
    en = np.linalg.norm(test_X, axis=1, keepdims=True); en[en == 0] = 1.0
    sim = (test_X / en) @ (train_X / tn).T
    return (train_y[sim.argmax(axis=1)] == test_y).mean()

perm = np.random.default_rng(0).permutation(len(features))
test_n = len(features) // 5
test_idx = perm[:test_n]; pool_idx = perm[test_n:]
X_test = features.iloc[test_idx].to_numpy(); y_test = labels.iloc[test_idx].to_numpy()
X_pool = features.iloc[pool_idx].to_numpy(); y_pool = labels.iloc[pool_idx].to_numpy()

acc_full = nn_accuracy(X_pool, y_pool, X_test, y_test)
budget = int(0.40 * len(X_pool))
gmm_pool = GaussianMixture(n_components=3, random_state=0).fit(X_pool)
probs_pool = gmm_pool.predict_proba(X_pool)
result = ActiveSampleSelector(strategy='hybrid', alpha=0.5).select(
    features=X_pool, budget=budget, probs=probs_pool,
)
acc_al = nn_accuracy(X_pool[result.selected_indices], y_pool[result.selected_indices], X_test, y_test)
print(f'full-cohort:     {acc_full:.3f}')
print(f'40% active:      {acc_al:.3f}')
print(f'ratio:           {acc_al / acc_full:.3f} (roadmap >= 0.90)')

## See also
- PSF v0.6 roadmap — Phase 3 F12: [docs/roadmap-v06-codeberg.md](../../docs/roadmap-v06-codeberg.md)
- F1 uncertainty layer: [docs/guides/uncertainty.md](../../docs/guides/uncertainty.md)